# 09 — Real Anonymized Speech: Content Evaluation

## Objective

Run the existing content-preservation framework on **real original/anonymized speech pairs** generated with an official VoicePrivacy Challenge 2026 baseline.

```
Original audio + anonymized audio
              ↓
          Faster-Whisper
              ↓
     original / anonymized transcripts
              ↓
    ┌─────────┼──────────┐
    ↓         ↓          ↓
Similarity   NLI   Semantic alignment → TPI
```

This notebook orchestrates the existing metrics. It does not redefine them.

## Experimental setup

- VoicePrivacy Challenge 2026 Track 1, LibriSpeech dev subset.
- Official B2 McAdams anonymization.
- Initial size: 100 paired utterances.
- The 100-pair manifest is built directly in this notebook from the VPC subset files.
- Same Faster-Whisper configuration for original and anonymized audio.
- Content metrics remain separate; no arbitrary combined loss yet.

The notebook therefore starts from the already-generated B2 WAV files and handles the pairing, ASR, and content evaluation end-to-end.


In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

# Robust to being launched from either the project root or notebooks/.
cwd = Path.cwd().resolve()
if (cwd / "data" / "vpc2026").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data" / "vpc2026").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root containing data/vpc2026."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "vpc2026"
RESULTS_DIR = PROJECT_ROOT / "data" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SUBSET_DIR = DATA_DIR / "subset_100"
ORIGINAL_DATA_DIR = DATA_DIR / "data" / "libri_dev"
ANON_WAV_DIR = DATA_DIR / "subset_100_b2" / "wav"

PAIR_MANIFEST = DATA_DIR / "pairs_100.csv"
TRANSCRIPT_OUTPUT = RESULTS_DIR / "vpc2026_b2_100_transcripts.csv"
METRICS_OUTPUT = RESULTS_DIR / "vpc2026_b2_100_content_metrics.csv"

print("Project root:", PROJECT_ROOT)
print("Subset:", SUBSET_DIR)
print("Original data:", ORIGINAL_DATA_DIR)
print("B2 anonymized WAVs:", ANON_WAV_DIR)


Project root: /Users/juni/Documents/interview-anonymization
Subset: /Users/juni/Documents/interview-anonymization/data/vpc2026/subset_100
Original data: /Users/juni/Documents/interview-anonymization/data/vpc2026/data/libri_dev
B2 anonymized WAVs: /Users/juni/Documents/interview-anonymization/data/vpc2026/subset_100_b2/wav


In [2]:
# Build the 100 original/anonymized pairs directly from the VPC subset.
# No separate make_pairs_100.py script is required.

WAV_SCP = SUBSET_DIR / "wav.scp"
UTT2SPK = SUBSET_DIR / "utt2spk"

for required_file in [WAV_SCP, UTT2SPK, ANON_WAV_DIR]:
    if not required_file.exists():
        raise FileNotFoundError(f"Missing required VPC output: {required_file}")

# Speaker mapping
utt2spk = {}
with open(UTT2SPK) as f:
    for line in f:
        utt, spk = line.strip().split()
        utt2spk[utt] = spk

# Pair original and B2-anonymized audio using the utterance ID.
pair_rows = []

with open(WAV_SCP) as f:
    for line in f:
        utt, rel_path = line.strip().split(maxsplit=1)

        # wav.scp paths are relative to data/vpc2026.
        original_path = DATA_DIR / rel_path
        anonymized_path = ANON_WAV_DIR / f"{utt}.wav"

        if not original_path.exists():
            raise FileNotFoundError(f"Missing original audio: {original_path}")
        if not anonymized_path.exists():
            raise FileNotFoundError(f"Missing anonymized audio: {anonymized_path}")
        if utt not in utt2spk:
            raise ValueError(f"Speaker ID missing for utterance: {utt}")

        pair_rows.append({
            "pair_id": utt,
            "original_path": str(original_path),
            "anonymized_path": str(anonymized_path),
            "speaker_id": utt2spk[utt],
        })

pairs_df = pd.DataFrame(pair_rows)

if len(pairs_df) != 100:
    raise ValueError(f"Expected exactly 100 pairs, found {len(pairs_df)}")

pairs_df.to_csv(PAIR_MANIFEST, index=False)

print(f"Created pair manifest: {PAIR_MANIFEST}")
print(f"Pairs loaded: {len(pairs_df)}")
print(f"Unique speakers: {pairs_df['speaker_id'].nunique()}")

display(pairs_df.head())


Created pair manifest: /Users/juni/Documents/interview-anonymization/data/vpc2026/pairs_100.csv
Pairs loaded: 100
Unique speakers: 40


,pair_id,original_path,anonymized_path,speaker_id
0,1462-170142-0001,/Users/juni/Documents/interview-anonymization/...,/Users/juni/Documents/interview-anonymization/...,1462
1,251-137823-0004,/Users/juni/Documents/interview-anonymization/...,/Users/juni/Documents/interview-anonymization/...,251
2,174-168635-0014,/Users/juni/Documents/interview-anonymization/...,/Users/juni/Documents/interview-anonymization/...,174
3,3170-137482-0040,/Users/juni/Documents/interview-anonymization/...,/Users/juni/Documents/interview-anonymization/...,3170
4,7850-73752-0004,/Users/juni/Documents/interview-anonymization/...,/Users/juni/Documents/interview-anonymization/...,7850


### Pairing checkpoint

At this point the notebook has verified the experimental input:

**100 original WAVs ↔ 100 B2-anonymized WAVs.**

The generated `pairs_100.csv` is the reproducible manifest used by all subsequent cells.


In [3]:
from faster_whisper import WhisperModel

ASR_MODEL_NAME = "base"
ASR_DEVICE = "cpu"
ASR_COMPUTE_TYPE = "int8"
ASR_LANGUAGE = "en"

asr_model = WhisperModel(
    ASR_MODEL_NAME,
    device=ASR_DEVICE,
    compute_type=ASR_COMPUTE_TYPE,
)

def transcribe_audio(audio_path):
    segments, info = asr_model.transcribe(
        str(audio_path),
        language=ASR_LANGUAGE,
    )
    return " ".join(
        segment.text.strip()
        for segment in segments
        if segment.text.strip()
    )

def transcribe_pairs(df):
    rows = []
    for _, row in df.iterrows():
        out = row.to_dict()
        out["original_transcript"] = transcribe_audio(row.original_path)
        out["anonymized_transcript"] = transcribe_audio(row.anonymized_path)
        rows.append(out)
    return pd.DataFrame(rows)

/opt/anaconda3/envs/interview-anon/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
if TRANSCRIPT_OUTPUT.exists():
    transcript_df = pd.read_csv(TRANSCRIPT_OUTPUT)
    print("Loaded cached transcripts:", len(transcript_df))
else:
    transcript_df = transcribe_pairs(pairs_df)
    transcript_df.to_csv(TRANSCRIPT_OUTPUT, index=False)
    print("Transcribed pairs:", len(transcript_df))

if len(transcript_df) != 100:
    raise ValueError(f"Expected 100 transcript pairs, found {len(transcript_df)}")

display(
    transcript_df[
        ["pair_id", "original_transcript", "anonymized_transcript"]
    ].head(10)
)


Loaded cached transcripts: 100


,pair_id,original_transcript,anonymized_transcript
0,1462-170142-0001,Emerging at Houston at half past three o'clock...,"Emerging at Euston, I have passed three o'cloc..."
1,251-137823-0004,Electronic equipment cascaded from the wall sh...,The electronic equipment cascaded from the wal...
2,174-168635-0014,Cousette was no longer in rags. She was in mou...,Cusette was no longer in rags. She was in mour...
3,3170-137482-0040,"But, although, believing fully in my oracles, ...","But, although, believing fully in my oracles, ..."
4,7850-73752-0004,"Ferdinand felt his freedom, as well as his hap...","Ferdinand fell to his freedom, as well as his ..."
5,1272-135031-0004,The king is flooded disgrace and your friends ...,"The king is planning to disgrace, and your fri..."
6,1993-147964-0000,"They sought about the house most of the day, a...","They sought about the house most of the day, a..."
7,6241-61943-0000,"On the second of the month, at 2 in the mornin...","On a second and a month, at 2 in the morning, ..."
8,3752-4943-0012,"Let him out, cried north again, stamping his f...","I don't know how to climb Earth again, stabbin..."
9,2035-147960-0015,I don't winked at me.,I don't want it to mean.


## 2. Load the existing semantic-alignment + TPI implementation

The implementation is already in notebook 07e. To avoid maintaining a second copy, this notebook executes only the implementation cells needed for real-data evaluation and skips the controlled examples.

In [5]:
import nbformat

METRIC_NOTEBOOK = PROJECT_ROOT / "notebooks" / "07e_semantic_alignment_TPI.ipynb"
if not METRIC_NOTEBOOK.exists():
    raise FileNotFoundError(METRIC_NOTEBOOK)

nb = nbformat.read(METRIC_NOTEBOOK, as_version=4)

metric_cell_indices = [2, 3, 5, 7, 9, 11, 12, 14, 16, 25]
metric_globals = globals()

for idx in metric_cell_indices:
    source = "".join(nb.cells[idx].source)
    exec(compile(source, f"07e_cell_{idx}", "exec"), metric_globals)

print("Loaded 07e metric implementation.")
print("SIMILARITY_THRESHOLD:", SIMILARITY_THRESHOLD)
print("TPI_STRATEGY:", TPI_STRATEGY, "N:", N)

PyTorch: 2.11.0


Loading weights: 100%|█████████████████████| 202/202 [00:00<00:00, 4591.12it/s]
[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model: bert-base-uncased
Device: cpu
Semantic similarity threshold: 0.8
TPI strategy: N-TA
N-TA N: 6
Loaded 07e metric implementation.
SIMILARITY_THRESHOLD: 0.8
TPI_STRATEGY: N-TA N: 6


## 3. Semantic similarity

The selected model from notebook 05c is `paraphrase-mpnet-base-v2`. Use the same model for both transcripts.

In [6]:
from sentence_transformers import SentenceTransformer

SIM_MODEL_NAME = "sentence-transformers/paraphrase-mpnet-base-v2"
sim_model = SentenceTransformer(SIM_MODEL_NAME)

original_embeddings = sim_model.encode(
    transcript_df["original_transcript"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
)
anonymized_embeddings = sim_model.encode(
    transcript_df["anonymized_transcript"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
)

transcript_df["semantic_similarity"] = np.sum(
    original_embeddings * anonymized_embeddings,
    axis=1,
)

print("Semantic similarity computed for", len(transcript_df), "pairs.")

Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 4948.28it/s]


Semantic similarity computed for 100 pairs.


## 4. NLI

NLI is used as a complementary content-preservation signal.

The exact model/configuration selected in notebook 06b is loaded below:
`MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli`.

For each original/anonymized transcript pair, we keep the three NLI probabilities
(entailment, neutral, contradiction) and the highest-probability predicted relation.

Do **not** combine NLI, TPI, and semantic similarity into one arbitrary loss at this stage.


In [7]:
# Selected NLI model from notebook 06b.
# Keep this cell isolated so the rest of the notebook can run before NLI is wired.
from transformers import AutoTokenizer, AutoModelForSequenceClassification
NLI_MODEL_NAME = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"

if NLI_MODEL_NAME is None:
    print("NLI is not wired yet. Continuing with semantic similarity + alignment/TPI.")
else:
    print(f"Loading NLI model: {NLI_MODEL_NAME}")

    nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
    nli_model = AutoModelForSequenceClassification.from_pretrained(
        NLI_MODEL_NAME
    )

    nli_model.to(DEVICE)
    nli_model.eval()

    print("NLI model loaded.")
    print("Labels:", nli_model.config.id2label)

Loading NLI model: MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli


Loading weights: 100%|█████████████████████| 394/394 [00:00<00:00, 5213.92it/s]

NLI model loaded.
Labels: {0: 'entailment', 1: 'neutral', 2: 'contradiction'}


In [8]:
def normalize_label(label):
    label = label.lower()

    if "entail" in label:
        return "entailment"
    if "contrad" in label:
        return "contradiction"
    if "neutral" in label:
        return "neutral"

    return label


def predict_nli(tokenizer, model, pairs, batch_size=8):
    all_probs = []
    labels_by_id = model.config.id2label

    for start in range(0, len(pairs), batch_size):
        batch = pairs[start:start + batch_size]

        s1 = [x[0] for x in batch]
        s2 = [x[1] for x in batch]

        inputs = tokenizer(
            s1,
            s2,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )

        inputs = {
            k: v.to(DEVICE)
            for k, v in inputs.items()
        }

        with torch.inference_mode():
            logits = model(**inputs).logits
            probs = torch.softmax(
                logits,
                dim=-1
            ).detach().cpu().numpy()

        all_probs.append(probs)

    probs = np.concatenate(all_probs, axis=0)

    rows = []

    for pair, p in zip(pairs, probs):

        by_label = {
            normalize_label(labels_by_id[i]): float(p[i])
            for i in range(len(p))
        }

        pred = max(
            by_label,
            key=by_label.get
        )

        rows.append({
            "sentence_1": pair[0],
            "sentence_2": pair[1],
            "entailment": by_label.get(
                "entailment", np.nan
            ),
            "neutral": by_label.get(
                "neutral", np.nan
            ),
            "contradiction": by_label.get(
                "contradiction", np.nan
            ),
            "predicted": pred,
        })

    return pd.DataFrame(rows)

In [9]:
nli_pairs = list(
    zip(
        transcript_df["original_transcript"],
        transcript_df["anonymized_transcript"]
    )
)

nli_results = predict_nli(
    nli_tokenizer,
    nli_model,
    nli_pairs
)

# Keep the pair identifier so NLI results can be merged
# back into the main metrics table.
nli_results.insert(
    0,
    "pair_id",
    transcript_df["pair_id"].values
)

display(nli_results.head())

,pair_id,sentence_1,sentence_2,entailment,neutral,contradiction,predicted
0,1462-170142-0001,Emerging at Houston at half past three o'clock...,"Emerging at Euston, I have passed three o'cloc...",0.203369,0.020889,0.775879,contradiction
1,251-137823-0004,Electronic equipment cascaded from the wall sh...,The electronic equipment cascaded from the wal...,0.998535,0.001343,0.000160,entailment
2,174-168635-0014,Cousette was no longer in rags. She was in mou...,Cusette was no longer in rags. She was in mour...,0.979004,0.012817,0.008057,entailment
3,3170-137482-0040,"But, although, believing fully in my oracles, ...","But, although, believing fully in my oracles, ...",0.549316,0.421631,0.028824,entailment
4,7850-73752-0004,"Ferdinand felt his freedom, as well as his hap...","Ferdinand fell to his freedom, as well as his ...",0.006145,0.986816,0.007004,neutral


## 5. Semantic alignment + TPI on real pairs

In [10]:
metric_rows = []

for _, row in transcript_df.iterrows():
    result = evaluate_content_pair(
        row.original_transcript,
        row.anonymized_transcript,
        similarity_threshold=SIMILARITY_THRESHOLD,
        tpi_strategy=TPI_STRATEGY,
        N=N,
    )

    metric_rows.append({
        "pair_id": row.pair_id,
        "TPI": result["TPI"],
        "L_TPI": result["L_TPI"],
        "TIC_original": result["tpi"]["TIC_original"],
        "information_lost": result["tpi"]["information_lost"],
        "n_affected_terms": len(result["alignment"]["affected_original_indices"]),
        "affected_terms": ", ".join(result["alignment"]["affected_original_terms"]),
    })

metrics_df = pd.DataFrame(metric_rows)

# Add semantic similarity.
metrics_df = metrics_df.merge(
    transcript_df[["pair_id", "semantic_similarity"]],
    on="pair_id",
    how="left",
)

# Add the NLI probabilities and predicted relation.
metrics_df = metrics_df.merge(
    nli_results[
        [
            "pair_id",
            "entailment",
            "neutral",
            "contradiction",
            "predicted",
        ]
    ],
    on="pair_id",
    how="left",
)

metrics_df.to_csv(METRICS_OUTPUT, index=False)

display(metrics_df.head(10))


,pair_id,TPI,L_TPI,TIC_original,information_lost,n_affected_terms,affected_terms,semantic_similarity,entailment,neutral,contradiction,predicted
0,1462-170142-0001,0.697676,0.302324,102.866319,31.098998,6,"Houston, at, half, past, it, once",0.853957,0.203369,0.020889,0.775879,contradiction
1,251-137823-0004,0.850371,0.149629,104.484821,15.634005,5,"Electronic, and, a, duty, chain",0.982177,0.998535,0.001343,0.000160,entailment
2,174-168635-0014,0.486268,0.513732,31.060297,15.956676,1,Cousette,0.819615,0.979004,0.012817,0.008057,entailment
3,3170-137482-0040,1.000000,0.000000,135.504357,0.000000,0,,0.997856,0.549316,0.421631,0.028824,entailment
4,7850-73752-0004,0.809683,0.190317,29.274006,5.571328,1,felt,0.891451,0.006145,0.986816,0.007004,neutral
5,1272-135031-0004,0.433205,0.566795,55.237204,31.308188,2,"flooded, disgrace",0.855182,0.014908,0.076233,0.908691,contradiction
6,1993-147964-0000,1.000000,0.000000,102.864029,0.000000,0,,0.971320,0.982422,0.016113,0.001477,entailment
7,6241-61943-0000,0.419391,0.580609,58.796735,34.137902,13,"On, the, second, of, the, month, precious, car...",0.660140,0.004211,0.010773,0.984863,contradiction
8,3752-4943-0012,0.000000,1.000000,51.218174,51.218174,9,"Let, him, out, cried, north, again, stamping, ...",0.429771,0.000221,0.000686,0.999023,contradiction
9,2035-147960-0015,0.000000,1.000000,37.466213,37.466213,5,"I, don't, winked, at, me",0.541139,0.003374,0.996094,0.000577,neutral


## 6. Descriptive statistics and cross-metric diagnostics

Keep semantic similarity, TPI, and NLI as separate content-preservation signals.

The NLI probabilities are continuous scores, while `predicted` is a categorical
summary of the highest-probability NLI relation. We therefore report:

1. descriptive statistics for semantic similarity, TPI, and the three NLI probabilities;
2. the distribution of predicted NLI relations;
3. pairwise correlations between the continuous metrics;
4. disagreement cases for qualitative inspection.

Do **not** combine the metrics into one arbitrary loss at this stage.


In [17]:
continuous_metrics = [
    "semantic_similarity",
    "TPI",
    "L_TPI",
    "n_affected_terms",
    "entailment",
    "neutral",
    "contradiction",
]

print("Number of pairs:", len(metrics_df))

display(
    metrics_df[continuous_metrics].describe()
)

print("Predicted NLI relation proportions:")
display(
    metrics_df["predicted"]
    .value_counts(normalize=True, dropna=False)
    .rename("proportion")
    .to_frame()
)


Number of pairs: 100


,semantic_similarity,TPI,L_TPI,n_affected_terms,entailment,neutral,contradiction
count,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
mean,0.831410,0.702596,0.297404,3.700000,0.554893,0.213181,0.231924
std,0.226919,0.356280,0.356280,6.137318,0.455144,0.339228,0.380047
min,0.121958,0.000000,0.000000,0.000000,0.000168,0.000659,0.000081
25%,0.755719,0.477946,0.000000,0.000000,0.008198,0.007444,0.000797
50%,0.938614,0.858516,0.141484,1.000000,0.830078,0.028099,0.005419
75%,1.000000,1.000000,0.522054,5.250000,0.987549,0.276489,0.362000
max,1.000000,1.000000,1.000000,40.000000,0.999023,0.996094,0.999023


Predicted NLI relation proportions:


,proportion
predicted,
entailment,0.58
contradiction,0.23
neutral,0.19


In [18]:
print("Correlation matrix:")

display(
    metrics_df[
        [
            "semantic_similarity",
            "TPI",
            "entailment",
            "neutral",
            "contradiction",
        ]
    ].corr()
)


Correlation matrix:


,semantic_similarity,TPI,entailment,neutral,contradiction
semantic_similarity,1.000000,0.882071,0.621587,-0.360087,-0.422996
TPI,0.882071,1.000000,0.611852,-0.353910,-0.416865
entailment,0.621587,0.611852,1.000000,-0.575753,-0.683627
neutral,-0.360087,-0.353910,-0.575753,1.000000,-0.203127
contradiction,-0.422996,-0.416865,-0.683627,-0.203127,1.000000


In [23]:
print("TPI quartiles:")
display(
    metrics_df["TPI"].quantile([0, 0.25, 0.50, 0.75, 1.00]).rename("TPI")
)

print("Semantic similarity quartiles:")
display(
    metrics_df["semantic_similarity"]
    .quantile([0, 0.25, 0.50, 0.75, 1.00])
    .rename("semantic_similarity")
)

print("NLI entailment quartiles:")
display(
    metrics_df["entailment"]
    .quantile([0, 0.25, 0.50, 0.75, 1.00])
    .rename("entailment")
)

print("NLI contradiction quartiles:")
display(
    metrics_df["contradiction"]
    .quantile([0, 0.25, 0.50, 0.75, 1.00])
    .rename("contradiction")
)


TPI quartiles:


0.00    0.000000
0.25    0.477946
0.50    0.858516
0.75    1.000000
1.00    1.000000
Name: TPI, dtype: float64

Semantic similarity quartiles:


0.00    0.121958
0.25    0.755719
0.50    0.938614
0.75    1.000000
1.00    1.000000
Name: semantic_similarity, dtype: float64

NLI entailment quartiles:


0.00    0.000168
0.25    0.008198
0.50    0.830078
0.75    0.987549
1.00    0.999023
Name: entailment, dtype: float64

NLI contradiction quartiles:


0.00    0.000081
0.25    0.000797
0.50    0.005419
0.75    0.362000
1.00    0.999023
Name: contradiction, dtype: float64

In [24]:
# Cases where semantic similarity is high but TPI is low.
# These are useful for checking whether the two metrics disagree on content preservation.

low_tpi_high_sim = metrics_df[
    (metrics_df["TPI"] < 0.50) &
    (metrics_df["semantic_similarity"] > 0.90)
].copy()

print(
    "Potential high-similarity / low-TPI cases:",
    len(low_tpi_high_sim),
)

display(
    transcript_df[
        ["pair_id", "original_transcript", "anonymized_transcript"]
    ].merge(
        low_tpi_high_sim[
            [
                "pair_id",
                "TPI",
                "semantic_similarity",
                "entailment",
                "neutral",
                "contradiction",
                "predicted",
            ]
        ],
        on="pair_id",
        how="inner",
    )[
        [
            "pair_id",
            "original_transcript",
            "anonymized_transcript",
            "semantic_similarity",
            "TPI",
            "entailment",
            "neutral",
            "contradiction",
            "predicted",
        ]
    ].head(10)
)


Potential high-similarity / low-TPI cases: 2


,pair_id,original_transcript,anonymized_transcript,semantic_similarity,TPI,entailment,neutral,contradiction,predicted
0,2078-142845-0024,1724,1724.,0.963714,0.000000,0.995117,0.004318,0.000685,entailment
1,1673-143397-0008,"At these blasts from a sound, the pillars of t...","At these blocks from the sounds, the pillars o...",0.904375,0.452982,0.896973,0.047943,0.055206,entailment


In [25]:
# Potential NLI / similarity disagreement:
# high semantic similarity but low entailment probability.

high_sim_low_entailment = metrics_df[
    (metrics_df["semantic_similarity"] >
     0.90) &
    (metrics_df["entailment"] <
     0.50)
].copy()

print(
    "Potential high-similarity / low-entailment cases:",
    len(high_sim_low_entailment),
)

display(
    transcript_df[
        ["pair_id", "original_transcript", "anonymized_transcript"]
    ].merge(
        high_sim_low_entailment[
            [
                "pair_id",
                "semantic_similarity",
                "TPI",
                "entailment",
                "neutral",
                "contradiction",
                "predicted",
            ]
        ],
        on="pair_id",
        how="inner",
    )[
        [
            "pair_id",
            "original_transcript",
            "anonymized_transcript",
            "semantic_similarity",
            "TPI",
            "entailment",
            "neutral",
            "contradiction",
            "predicted",
        ]
    ].head(10)
)


Potential high-similarity / low-entailment cases: 7


,pair_id,original_transcript,anonymized_transcript,semantic_similarity,TPI,entailment,neutral,contradiction,predicted
0,6319-57405-0005,"The day is coming,"" said Pometheus, when Jupit...","The day is coming, sepomethious, when Jupiter ...",0.902973,0.776897,0.043091,0.943359,0.013458,neutral
1,251-137823-0000,"I'll be glad to try, sir,"" he replied.","I'll be glad to try, sir. You replied.",0.916970,0.891724,0.114441,0.847168,0.038330,neutral
2,3853-163249-0012,"Then they went back to their work, little drea...","Then they went back to their work, little drea...",0.917260,0.866662,0.001323,0.017014,0.981445,contradiction
3,3576-138058-0000,"Master and man dismounted from their beasts, a...","Master and man dismounted from their beasts, a...",0.944841,0.757205,0.008240,0.004017,0.987793,contradiction
4,3000-15664-0001,"Arctic beauty and desolation, with their bless...","Arctic beauty and desolation, with their bless...",0.995066,0.958548,0.051208,0.339844,0.608887,contradiction
5,2277-149897-0025,He was quite certain now that she knew he was ...,He was quite certain now that she knew he was ...,0.932387,0.650445,0.004478,0.911621,0.083740,neutral
6,3576-138058-0014,"Don Quixote was on foot, with his horse unbrid...",Don Quixote was on foot with his horse unbridl...,0.976126,0.787900,0.289551,0.255371,0.455078,contradiction


In [30]:
# Optional compact view of the main content-preservation metrics.
display(
    metrics_df[
        [
            "pair_id",
            "semantic_similarity",
            "TPI",
            "entailment",
            "neutral",
            "contradiction",
            "predicted",
        ]
    ].head(10)
)


,pair_id,semantic_similarity,TPI,entailment,neutral,contradiction,predicted
0,1462-170142-0001,0.853957,0.697676,0.203369,0.020889,0.775879,contradiction
1,251-137823-0004,0.982177,0.850371,0.998535,0.001343,0.000160,entailment
2,174-168635-0014,0.819615,0.486268,0.979004,0.012817,0.008057,entailment
3,3170-137482-0040,0.997856,1.000000,0.549316,0.421631,0.028824,entailment
4,7850-73752-0004,0.891451,0.809683,0.006145,0.986816,0.007004,neutral
5,1272-135031-0004,0.855182,0.433205,0.014908,0.076233,0.908691,contradiction
6,1993-147964-0000,0.971320,1.000000,0.982422,0.016113,0.001477,entailment
7,6241-61943-0000,0.660140,0.419391,0.004211,0.010773,0.984863,contradiction
8,3752-4943-0012,0.429771,0.000000,0.000221,0.000686,0.999023,contradiction
9,2035-147960-0015,0.541139,0.000000,0.003374,0.996094,0.000577,neutral
